# 02 · Modelo de fraude — Argly
Scoring **híbrido**: reglas (rúbrica + RF gates) + ML supervisado (RandomForest) + anomalías (Isolation Forest). El desglose ES la lista de contribuciones.

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

In [2]:
from src.ingestion.generate_synthetic import generar
from src.features.build_features import construir_features
from src.models.ml_model import entrenar, explicacion_global
dfs = generar(seed=42)
F = construir_features(dfs)
modelo, prob = entrenar(F)
F.shape

(1465, 25)

## Explicabilidad global del modelo (SHAP si está instalado)

In [3]:
exp = explicacion_global(modelo, F)
print('método:', exp['metodo'])
exp['importancias']

/home/sebbarco/projects/argly/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


método: shap


{'dias_desde_inicio_poliza': 0.0607,
 'ratio_monto_suma': 0.0594,
 'dias_entre_ocurrencia_reporte': 0.0406,
 'distancia_geo_km': 0.0192,
 'es_robo': 0.0184,
 'clima_inconsistente': 0.0155,
 'historial_siniestros_asegurado': 0.0143,
 'freq_solo_rc': 0.0107,
 'narrativa_repetidos': 0.0107,
 'evento_sin_tercero': 0.0101,
 'doc_inconsistente': 0.0097,
 'documentos_completos': 0.0091,
 'doc_no_entregado': 0.0087,
 'freq_conductor': 0.0085,
 'es_ptxrb': 0.0071,
 'proveedor_en_lista': 0.0066,
 'freq_proveedor': 0.006,
 'freq_vehiculo': 0.0032,
 'asegurado_en_lista': 0.0}

## Bandeja priorizada (score 0-100 + semáforo)

In [4]:
from src.pipeline import construir_bandeja
b = construir_bandeja(dfs)
b['nivel'].value_counts()

nivel
VERDE       1150
AMARILLO     196
ROJO         119
Name: count, dtype: int64

In [5]:
b[['id_siniestro','score','nivel','motivo_principal','monto_reclamado']].head(10)

,id_siniestro,score,nivel,motivo_principal,monto_reclamado
0,SIN001312,100,ROJO,RF-02 Adulteración documental,4563.52
1,SIN001319,100,ROJO,RF-02 Adulteración documental,1438.51
2,SIN001302,100,ROJO,RF-02 Adulteración documental,2965.89
3,SIN001263,98,ROJO,RF-03 Lista Restrictiva,11057.76
4,SIN001308,96,ROJO,RF-02 Adulteración documental,13180.84
5,SIN001306,95,ROJO,RF-02 Adulteración documental,3927.06
6,SIN001311,95,ROJO,RF-02 Adulteración documental,7273.64
7,SIN001313,95,ROJO,RF-02 Adulteración documental,2206.05
8,SIN001304,95,ROJO,RF-02 Adulteración documental,3876.20
9,SIN001324,95,ROJO,RF-02 Adulteración documental,3558.31
